工具是Agent的最重要的能力之一：

- Agent可以直接调用工具并拿到返回值，是围绕LLM的一层能力包装
- LLM发起工具调用之类，但是无法拿到工具的返回值


## 工具直接调用

LangChain中的工具抽象也是runnable的，有invoke方法直接调用

In [6]:
from langchain_core.tools import tool
@tool
def get_weather(city: str) -> str:
    """
        获取指定城市的天气信息
        参数:
            city: 城市名称，如"北京"、"上海"
        返回:
            天气信息字符串
    """
    return city + "晴天，温度 15°C"

result = get_weather.invoke({"city": "北京"})
print(result) 
print(type(result))

北京晴天，温度 15°C
<class 'str'>


## 与模型绑定

工具可以绑定在模型上，模型就知道自己可以调用哪些工具，需要时发起tool_call。但是模型不会执行工具，需要我们手动执行。


In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rich_print
import os
# 从.env文件中加载环境变量
load_dotenv(override=True)

model = init_chat_model(model = "deepseek-v4-flash") # 配置文件中有官方维护的模型的API信息时不需要指定

#定义工具
@tool
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    return city + "晴天，温度 15°C"
# 绑定工具
model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("北京天气如何？")
rich_print(response)


AIMessage(
    content='',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': '用户想知道北京的天气。我可以使用 get_weather 工具来查询北京的天气。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 62,
            'prompt_tokens': 275,
            'total_tokens': 337,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 17,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 275
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
        'id': '2c13d7b7-b435-420e-b7b9-d8845a2f5e2c',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--019f2295-0aa0-77f2-87c4-84b9708ada89-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '北京'},
            'id': 'call_00_7nYvUB4TZpqL22xWdATr7835',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 275,
        'output_tokens': 62,
        'total_tokens': 337,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 17}
    }
)

In [4]:
rich_print(response.content_blocks)


[
    {'type': 'reasoning', 'reasoning': '用户想知道北京的天气。我可以使用 get_weather 工具来查询北京的天气。'},
    {
        'type': 'tool_call',
        'id': 'call_00_7nYvUB4TZpqL22xWdATr7835',
        'name': 'get_weather',
        'args': {'city': '北京'}
    }
]

# 与模型绑定并调用

In [ ]:
from langchain.messages import HumanMessage, ToolMessage
from langchain.tools import tool
from rich import print as rich_print

@tool
def get_weather(city: str):
    """获取天气的工具"""
    return f"{city}天气晴朗~"

# 将模型和工具绑定
model_with_tools = model.bind_tools([get_weather])

messages = [
    HumanMessage("今天武汉天气如何")
]

# 模型生成调用工具请求
response = model_with_tools.invoke(messages)

# 添加AIMessage
messages.append(response)

tool_calls = response.tool_calls # tool_calls是一个列表，每个元素是一个TypedDict字典，包含工具调用的信息
rich_print(tool_calls)

[{'name': 'get_weather', 'args': {'city': '武汉'}, 'id': 'call_00_L01Q6SsWmikXxxkNBWdl0140', 'type': 'tool_call'}]

In [ ]:
print(type(tool_calls[0])) # TypedDict


<class 'dict'>


In [ ]:
for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        # 直接传递的tool_calls，TypedDict列表而不是工具函数参数JSON，Langchain会把返回值包装为ToolMessage类型消息
        tool_response = get_weather.invoke(tool_call) 
        print(type(tool_response))
        messages.append(tool_response)

final_response = model_with_tools.invoke(messages)
print(f"final_response: \n{final_response}")

<class 'langchain_core.messages.tool.ToolMessage'>
final_response: 
content='今天武汉天气晴朗！☀️ 是个好天气，适合出门活动哦～' additional_kwargs={'refusal': None, 'reasoning_content': '工具返回了武汉的天气信息。让我告诉用户。'} response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 354, 'total_tokens': 385, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 12, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 98}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'a4c32cff-73a4-4889-8291-69d5831edd25', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f229c-c6eb-7492-8ebe-1aba7262aa1d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 354, 'output_tokens': 31, 'total_tokens': 385, 'input_token_details': {'cache_read'